# 🛒 Análise de Faturamento — Olist E-Commerce Brasil

Análise exploratória completa do dataset **Olist Brazilian E-Commerce**, respondendo perguntas reais de negócio com SQL, Pandas e Plotly.

## Perguntas de negócio respondidas
1. Qual a evolução da receita ao longo do tempo?
2. Quais categorias geram mais faturamento?
3. Quais estados concentram mais pedidos e receita?
4. Qual o ticket médio por categoria?
5. Como está a satisfação dos clientes?
6. Qual o tempo médio de entrega e onde estamos atrasando?

---
**Dataset:** [Olist Brazilian E-Commerce — Kaggle](https://www.kaggle.com/datasets/olistbr/brazilian-ecommerce)  
**Período:** 2016 a 2018

## 1. Setup e Importações

In [ ]:
import sqlite3
import os
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

pd.set_option('display.float_format', '{:.2f}'.format)
pd.set_option('display.max_columns', 50)

DATA_DIR = '../data'
DB_PATH  = os.path.join(DATA_DIR, 'olist.db')

print('Setup concluído!')

## 2. Carregamento dos Dados e Criação do Banco SQLite

> **Como obter os dados:** Acesse [Kaggle — Olist](https://www.kaggle.com/datasets/olistbr/brazilian-ecommerce), baixe os CSVs e coloque-os na pasta `data/`.

In [ ]:
# Mapeamento: nome da tabela → arquivo CSV
tabelas = {
    'orders':       'olist_orders_dataset.csv',
    'order_items':  'olist_order_items_dataset.csv',
    'customers':    'olist_customers_dataset.csv',
    'products':     'olist_products_dataset.csv',
    'sellers':      'olist_sellers_dataset.csv',
    'reviews':      'olist_order_reviews_dataset.csv',
    'payments':     'olist_order_payments_dataset.csv',
    'categories':   'product_category_name_translation.csv',
    'geolocation':  'olist_geolocation_dataset.csv',
}

conn = sqlite3.connect(DB_PATH)

for tabela, arquivo in tabelas.items():
    caminho = os.path.join(DATA_DIR, arquivo)
    if os.path.exists(caminho):
        df = pd.read_csv(caminho)
        df.to_sql(tabela, conn, if_exists='replace', index=False)
        print(f'✅ {tabela:15s} — {len(df):>7,} linhas')
    else:
        print(f'⚠️  {arquivo} não encontrado. Coloque os CSVs na pasta data/')

print('\nBanco SQLite criado em:', DB_PATH)

## 3. Visão Geral — KPIs Principais

In [ ]:
query_kpis = """
SELECT
    COUNT(DISTINCT o.order_id)                             AS total_pedidos,
    COUNT(DISTINCT o.customer_id)                          AS total_clientes,
    ROUND(SUM(oi.price + oi.freight_value), 2)             AS receita_total,
    ROUND(AVG(oi.price + oi.freight_value), 2)             AS ticket_medio,
    ROUND(AVG(r.review_score), 2)                          AS nota_media,
    COUNT(DISTINCT oi.seller_id)                           AS total_vendedores
FROM orders o
JOIN order_items  oi ON o.order_id = oi.order_id
LEFT JOIN reviews  r ON o.order_id = r.order_id
WHERE o.order_status = 'delivered'
"""

kpis = pd.read_sql(query_kpis, conn)

print('=' * 50)
print('📊 KPIs GERAIS — Pedidos Entregues')
print('=' * 50)
print(f"Total de Pedidos:   {kpis['total_pedidos'].iloc[0]:>10,}")
print(f"Total de Clientes:  {kpis['total_clientes'].iloc[0]:>10,}")
print(f"Receita Total:      R$ {kpis['receita_total'].iloc[0]:>10,.2f}")
print(f"Ticket Médio:       R$ {kpis['ticket_medio'].iloc[0]:>10,.2f}")
print(f"Nota Média:         {kpis['nota_media'].iloc[0]:>10.2f} / 5.0")
print(f"Total Vendedores:   {kpis['total_vendedores'].iloc[0]:>10,}")

## 4. Evolução da Receita Mensal

In [ ]:
query_receita_mensal = """
SELECT
    STRFTIME('%Y-%m', o.order_purchase_timestamp) AS mes,
    COUNT(DISTINCT o.order_id)                     AS pedidos,
    ROUND(SUM(oi.price + oi.freight_value), 2)     AS receita
FROM orders o
JOIN order_items oi ON o.order_id = oi.order_id
WHERE o.order_status = 'delivered'
  AND o.order_purchase_timestamp IS NOT NULL
GROUP BY mes
ORDER BY mes
"""

receita_mensal = pd.read_sql(query_receita_mensal, conn)
receita_mensal['mes'] = pd.to_datetime(receita_mensal['mes'])

# Remover meses incompletos nas bordas
receita_mensal = receita_mensal[
    (receita_mensal['mes'] >= '2017-01-01') &
    (receita_mensal['mes'] <= '2018-08-01')
]

fig = make_subplots(rows=2, cols=1, shared_xaxes=True,
                    subplot_titles=('Receita Mensal (R$)', 'Volume de Pedidos'),
                    vertical_spacing=0.12)

fig.add_trace(go.Scatter(
    x=receita_mensal['mes'], y=receita_mensal['receita'],
    mode='lines+markers', name='Receita',
    line=dict(color='#4f46e5', width=2),
    fill='tozeroy', fillcolor='rgba(79,70,229,0.08)'
), row=1, col=1)

fig.add_trace(go.Bar(
    x=receita_mensal['mes'], y=receita_mensal['pedidos'],
    name='Pedidos', marker_color='#06b6d4'
), row=2, col=1)

fig.update_layout(
    title='Evolução da Receita e Volume de Pedidos — 2017/2018',
    height=500, showlegend=False,
    plot_bgcolor='white', paper_bgcolor='white'
)
fig.update_yaxes(tickprefix='R$ ', row=1, col=1)
fig.show()

## 5. Top 10 Categorias por Faturamento

In [ ]:
query_categorias = """
SELECT
    COALESCE(c.product_category_name_english, p.product_category_name, 'Sem Categoria') AS categoria,
    COUNT(DISTINCT o.order_id)                  AS pedidos,
    ROUND(SUM(oi.price), 2)                     AS receita,
    ROUND(AVG(oi.price), 2)                     AS ticket_medio,
    ROUND(AVG(r.review_score), 2)               AS nota_media
FROM orders o
JOIN order_items  oi ON o.order_id  = oi.order_id
JOIN products      p ON oi.product_id = p.product_id
LEFT JOIN categories c ON p.product_category_name = c.product_category_name
LEFT JOIN reviews    r ON o.order_id = r.order_id
WHERE o.order_status = 'delivered'
GROUP BY categoria
ORDER BY receita DESC
LIMIT 10
"""

categorias = pd.read_sql(query_categorias, conn)

fig = px.bar(
    categorias.sort_values('receita'),
    x='receita', y='categoria',
    orientation='h',
    text='receita',
    color='nota_media',
    color_continuous_scale='RdYlGn',
    range_color=[1, 5],
    labels={'receita': 'Receita (R$)', 'categoria': 'Categoria', 'nota_media': 'Nota Média'},
    title='Top 10 Categorias por Faturamento (cor = satisfação do cliente)'
)
fig.update_traces(texttemplate='R$ %{text:,.0f}', textposition='outside')
fig.update_layout(height=450, plot_bgcolor='white', paper_bgcolor='white')
fig.show()

print('\nTabela detalhada:')
display(categorias.rename(columns={
    'categoria': 'Categoria',
    'pedidos': 'Pedidos',
    'receita': 'Receita (R$)',
    'ticket_medio': 'Ticket Médio',
    'nota_media': 'Nota Média'
}))

## 6. Distribuição Geográfica por Estado

In [ ]:
query_estados = """
SELECT
    cu.customer_state                           AS estado,
    COUNT(DISTINCT o.order_id)                  AS pedidos,
    ROUND(SUM(oi.price + oi.freight_value), 2)  AS receita,
    ROUND(AVG(oi.price + oi.freight_value), 2)  AS ticket_medio
FROM orders o
JOIN order_items oi ON o.order_id      = oi.order_id
JOIN customers   cu ON o.customer_id   = cu.customer_id
WHERE o.order_status = 'delivered'
GROUP BY estado
ORDER BY receita DESC
"""

estados = pd.read_sql(query_estados, conn)

fig = make_subplots(rows=1, cols=2,
                    subplot_titles=('Top 10 Estados por Receita', 'Ticket Médio por Estado'))

top10 = estados.head(10)

fig.add_trace(go.Bar(
    x=top10['estado'], y=top10['receita'],
    marker_color='#4f46e5', name='Receita'
), row=1, col=1)

fig.add_trace(go.Bar(
    x=top10['estado'], y=top10['ticket_medio'],
    marker_color='#06b6d4', name='Ticket Médio'
), row=1, col=2)

fig.update_layout(
    title='Análise Geográfica — Pedidos por Estado',
    height=420, showlegend=False,
    plot_bgcolor='white', paper_bgcolor='white'
)
fig.update_yaxes(tickprefix='R$ ')
fig.show()

print(f"\nSP + RJ + MG concentram {estados[estados['estado'].isin(['SP','RJ','MG'])]['receita'].sum() / estados['receita'].sum() * 100:.1f}% da receita total")

## 7. Análise de Satisfação dos Clientes

In [ ]:
query_avaliacoes = """
SELECT
    review_score             AS nota,
    COUNT(*)                 AS quantidade,
    ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (), 1) AS percentual
FROM reviews
GROUP BY review_score
ORDER BY review_score
"""

avaliacoes = pd.read_sql(query_avaliacoes, conn)

cores = {
    1: '#ef4444', 2: '#f97316', 3: '#eab308',
    4: '#84cc16', 5: '#22c55e'
}

fig = go.Figure(go.Bar(
    x=avaliacoes['nota'].astype(str) + ' ⭐',
    y=avaliacoes['quantidade'],
    text=avaliacoes['percentual'].astype(str) + '%',
    textposition='outside',
    marker_color=[cores[n] for n in avaliacoes['nota']]
))

fig.update_layout(
    title='Distribuição das Avaliações dos Clientes',
    xaxis_title='Nota', yaxis_title='Quantidade de Avaliações',
    height=400, plot_bgcolor='white', paper_bgcolor='white'
)
fig.show()

positivas = avaliacoes[avaliacoes['nota'] >= 4]['percentual'].sum()
print(f"\n✅ {positivas:.1f}% das avaliações são positivas (nota 4 ou 5)")

## 8. Análise de Tempo de Entrega

In [ ]:
query_entrega = """
SELECT
    cu.customer_state AS estado,
    ROUND(AVG(
        JULIANDAY(o.order_delivered_customer_date) -
        JULIANDAY(o.order_purchase_timestamp)
    ), 1) AS dias_entrega_real,
    ROUND(AVG(
        JULIANDAY(o.order_estimated_delivery_date) -
        JULIANDAY(o.order_purchase_timestamp)
    ), 1) AS dias_entrega_estimado,
    COUNT(*) AS pedidos
FROM orders o
JOIN customers cu ON o.customer_id = cu.customer_id
WHERE o.order_status = 'delivered'
  AND o.order_delivered_customer_date IS NOT NULL
GROUP BY estado
HAVING pedidos >= 50
ORDER BY dias_entrega_real DESC
"""

entrega = pd.read_sql(query_entrega, conn)
entrega['atraso_medio'] = entrega['dias_entrega_real'] - entrega['dias_entrega_estimado']

fig = go.Figure()

fig.add_trace(go.Bar(
    name='Entrega Real', x=entrega['estado'],
    y=entrega['dias_entrega_real'], marker_color='#4f46e5'
))
fig.add_trace(go.Bar(
    name='Estimado', x=entrega['estado'],
    y=entrega['dias_entrega_estimado'], marker_color='#e5e7eb'
))

fig.update_layout(
    barmode='group',
    title='Tempo Médio de Entrega Real vs. Estimado por Estado (dias)',
    xaxis_title='Estado', yaxis_title='Dias',
    height=450, plot_bgcolor='white', paper_bgcolor='white'
)
fig.show()

media_geral = pd.read_sql("""
    SELECT ROUND(AVG(
        JULIANDAY(order_delivered_customer_date) -
        JULIANDAY(order_purchase_timestamp)
    ), 1) AS media FROM orders
    WHERE order_status='delivered' AND order_delivered_customer_date IS NOT NULL
""", conn)['media'].iloc[0]

print(f"⏱️  Tempo médio de entrega geral: {media_geral} dias")
antecipados = (entrega['atraso_medio'] < 0).sum()
print(f"✅ {antecipados}/{len(entrega)} estados entregam antes do prazo estimado em média")

## 9. Análise de Formas de Pagamento

In [ ]:
query_pagamentos = """
SELECT
    payment_type                                AS forma_pagamento,
    COUNT(DISTINCT order_id)                    AS pedidos,
    ROUND(SUM(payment_value), 2)                AS valor_total,
    ROUND(AVG(payment_installments), 1)         AS parcelas_medias
FROM payments
WHERE payment_type != 'not_defined'
GROUP BY payment_type
ORDER BY pedidos DESC
"""

pagamentos = pd.read_sql(query_pagamentos, conn)
pagamentos['forma_pagamento'] = pagamentos['forma_pagamento'].replace({
    'credit_card': 'Cartão de Crédito',
    'boleto': 'Boleto',
    'voucher': 'Voucher',
    'debit_card': 'Cartão de Débito'
})

fig = make_subplots(rows=1, cols=2, specs=[[{'type':'pie'}, {'type':'bar'}]],
                    subplot_titles=('Participação por Pedidos', 'Parcelas Médias'))

fig.add_trace(go.Pie(
    labels=pagamentos['forma_pagamento'],
    values=pagamentos['pedidos'],
    hole=0.4,
    marker_colors=['#4f46e5','#06b6d4','#84cc16','#f97316']
), row=1, col=1)

fig.add_trace(go.Bar(
    x=pagamentos['forma_pagamento'],
    y=pagamentos['parcelas_medias'],
    marker_color='#4f46e5', name='Parcelas'
), row=1, col=2)

fig.update_layout(
    title='Análise de Formas de Pagamento',
    height=420, plot_bgcolor='white', paper_bgcolor='white'
)
fig.show()

## 10. Conclusões e Insights

### 📈 Receita
- A Olist apresentou crescimento consistente de 2017 para 2018, com pico em novembro (Black Friday)
- As categorias **bed_bath_table**, **health_beauty** e **computers_accessories** lideram o faturamento

### 🗺️ Distribuição Geográfica
- **SP, RJ e MG** concentram a maior parte da receita — padrão esperado dado o tamanho dos mercados
- Estados do Norte e Nordeste têm ticket médio comparável mas volume menor

### ⭐ Satisfação
- Mais de **75% das avaliações** são positivas (nota 4 ou 5)
- Categorias com alto faturamento nem sempre têm as melhores notas — oportunidade de melhoria em CS

### ⏱️ Entrega
- Tempo médio de entrega é satisfatório para a maioria dos estados
- Regiões Norte e Nordeste apresentam maior tempo de entrega — logística é o gargalo

### 💳 Pagamento
- **Cartão de crédito** domina com folga — parcelamento é um driver importante de conversão no Brasil

---
*Análise realizada por Augusto Matos | Dataset: Olist Brazilian E-Commerce (Kaggle)*

In [ ]:
# Fechar conexão com o banco
conn.close()
print('Análise concluída. Conexão com o banco encerrada.')